# Lab 4: Knowledge, RAG & AI Search (15 min)

## Objetivos
- Entender o que é RAG (Retrieval-Augmented Generation)
- Criar embeddings de documentos
- Indexar documentos no Azure AI Search
- Criar um agente com grounding (conhecimento)

## 📖 O que é RAG?

**RAG** = Retrieval-Augmented Generation

É a técnica de dar "conhecimento extra" ao modelo, pesquisando em documentos antes de responder.

```
Pergunta do utilizador
        │
        ▼
┌─────────────────┐
│  1. PESQUISAR    │ ← Encontra documentos relevantes
│  (AI Search)     │   nos teus dados
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  2. AUMENTAR     │ ← Junta os documentos à pergunta
│  (contexto)      │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  3. GERAR        │ ← O modelo responde com base nos
│  (LLM)           │   documentos encontrados
└─────────────────┘
```

### Porquê RAG?
- O modelo **não conhece** os teus dados privados
- Fine-tuning é caro e demorado
- RAG é **dinâmico** — atualiza dados sem re-treinar o modelo

In [ ]:
# Setup
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from openai import AzureOpenAI

openai_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview"),
)

DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")
EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002")

print("✅ Setup concluído!")

## 💻 Exercício 4.1: Criar Embeddings

**Embeddings** são representações numéricas (vetores) do texto. Textos semelhantes têm vetores próximos.

In [ ]:
# Criar embedding de uma frase
resultado = openai_client.embeddings.create(
    model=EMBEDDING_DEPLOYMENT,
    input="Quantos dias de férias tenho direito?",
)

embedding = resultado.data[0].embedding
print(f"📐 Dimensões do embedding: {len(embedding)}")
print(f"📊 Primeiros 10 valores: {embedding[:10]}")

In [ ]:
# Comparar similaridade entre frases
import numpy as np

frases = [
    "Quantos dias de férias tenho?",
    "Como posso marcar as minhas férias?",
    "Qual é a receita de bacalhau à brás?",
    "Quantos dias de descanso me pertencem por ano?",
]

embeddings = openai_client.embeddings.create(
    model=EMBEDDING_DEPLOYMENT,
    input=frases,
)

vetores = [e.embedding for e in embeddings.data]

# Calcular similaridade coseno
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("📊 Similaridade entre frases (vs primeira frase):")
print(f"   Base: '{frases[0]}'\n")
for i in range(1, len(frases)):
    sim = cosine_similarity(vetores[0], vetores[i])
    barra = "█" * int(sim * 30)
    print(f"   {sim:.3f} {barra} '{frases[i]}'")

## 💻 Exercício 4.2: Indexar Documentos no AI Search

Vamos carregar o documento de exemplo e indexá-lo no Azure AI Search.

In [ ]:
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SimpleField,
    SearchableField,
    SearchField,
    SearchFieldDataType,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
)
from azure.core.credentials import AzureKeyCredential

search_endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
search_key = os.getenv("AZURE_SEARCH_KEY")
index_name = os.getenv("AZURE_SEARCH_INDEX", "workshop-index")

# Criar o índice
index_client = SearchIndexClient(
    endpoint=search_endpoint,
    credential=AzureKeyCredential(search_key),
)

index = SearchIndex(
    name=index_name,
    fields=[
        SimpleField(name="id", type=SearchFieldDataType.String, key=True),
        SearchableField(name="content", type=SearchFieldDataType.String),
        SearchableField(name="title", type=SearchFieldDataType.String),
        SearchField(
            name="embedding",
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            searchable=True,
            vector_search_dimensions=1536,
            vector_search_profile_name="my-vector-profile",
        ),
    ],
    vector_search=VectorSearch(
        algorithms=[HnswAlgorithmConfiguration(name="my-hnsw")],
        profiles=[VectorSearchProfile(name="my-vector-profile", algorithm_configuration_name="my-hnsw")],
    ),
)

result = index_client.create_or_update_index(index)
print(f"✅ Índice '{result.name}' criado/atualizado!")

In [ ]:
# Ler o documento de exemplo e dividir em chunks
with open("../data/documentos/exemplo.md", "r", encoding="utf-8") as f:
    documento = f.read()

# Dividir por secções (## como separador)
seccoes = documento.split("### ")
chunks = []
for i, seccao in enumerate(seccoes):
    if seccao.strip():
        chunks.append({
            "id": f"doc-{i}",
            "title": seccao.split("\n")[0].strip(),
            "content": seccao.strip(),
        })

print(f"📄 Documento dividido em {len(chunks)} chunks:")
for c in chunks:
    print(f"   - {c['title'][:60]}")

In [ ]:
# Gerar embeddings e carregar no índice
search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=index_name,
    credential=AzureKeyCredential(search_key),
)

documentos_indexar = []
for chunk in chunks:
    emb = openai_client.embeddings.create(
        model=EMBEDDING_DEPLOYMENT,
        input=chunk["content"],
    )
    chunk["embedding"] = emb.data[0].embedding
    documentos_indexar.append(chunk)

result = search_client.upload_documents(documents=documentos_indexar)
print(f"✅ {len(result)} documentos indexados com sucesso!")

## 💻 Exercício 4.3: Pesquisar com Vector Search

In [ ]:
from azure.search.documents.models import VectorizableTextQuery

# Pesquisa híbrida (texto + vetor)
pergunta = "Quantos dias de férias tenho direito?"

# Gerar embedding da pergunta
q_emb = openai_client.embeddings.create(
    model=EMBEDDING_DEPLOYMENT,
    input=pergunta,
).data[0].embedding

# Pesquisar
from azure.search.documents.models import VectorizedQuery

resultados = search_client.search(
    search_text=pergunta,
    vector_queries=[
        VectorizedQuery(
            vector=q_emb,
            k_nearest_neighbors=3,
            fields="embedding",
        )
    ],
    top=3,
)

print(f"🔍 Pergunta: '{pergunta}'\n")
contexto_docs = []
for r in resultados:
    print(f"  📄 {r['title']} (score: {r['@search.score']:.3f})")
    print(f"     {r['content'][:100]}...\n")
    contexto_docs.append(r["content"])

## 💻 Exercício 4.4: RAG Completo — Pesquisar + Gerar Resposta

In [ ]:
# RAG: usar os documentos encontrados como contexto para o LLM
contexto = "\n---\n".join(contexto_docs)

response = openai_client.chat.completions.create(
    model=DEPLOYMENT,
    messages=[
        {
            "role": "system",
            "content": f"""Tu és um assistente de RH da empresa Contoso.
Responde APENAS com base nos documentos fornecidos.
Se a informação não estiver nos documentos, diz que não tens essa informação.

DOCUMENTOS:
{contexto}""",
        },
        {"role": "user", "content": pergunta},
    ],
    temperature=0.1,
    max_tokens=300,
)

print(f"👤 Pergunta: {pergunta}\n")
print(f"🤖 Resposta RAG:\n{response.choices[0].message.content}")

In [ ]:
# Testar com outra pergunta
pergunta2 = "O que acontece se eu ficar doente durante as férias?"

q_emb2 = openai_client.embeddings.create(model=EMBEDDING_DEPLOYMENT, input=pergunta2).data[0].embedding

resultados2 = search_client.search(
    search_text=pergunta2,
    vector_queries=[VectorizedQuery(vector=q_emb2, k_nearest_neighbors=3, fields="embedding")],
    top=3,
)

contexto2 = "\n---\n".join([r["content"] for r in resultados2])

response2 = openai_client.chat.completions.create(
    model=DEPLOYMENT,
    messages=[
        {
            "role": "system",
            "content": f"""Tu és um assistente de RH da empresa Contoso.
Responde APENAS com base nos documentos fornecidos.

DOCUMENTOS:
{contexto2}""",
        },
        {"role": "user", "content": pergunta2},
    ],
    temperature=0.1,
    max_tokens=300,
)

print(f"👤 {pergunta2}\n")
print(f"🤖 {response2.choices[0].message.content}")

## ✅ Resumo

Neste lab aprendeste:
- O que é RAG e porquê é útil
- Como criar embeddings com Azure OpenAI
- Como criar e popular um índice no AI Search
- Como fazer pesquisa vetorial
- Como implementar o padrão RAG completo

**Próximo:** [Lab 5 - Workflows](lab05-workflows.ipynb)